# 02 · Piloto: transcrição com Whisper e WER por sotaque

Fluxo completo em uma amostra pequena do CORAA (conjunto de teste):

1. Baixar os áudios do conjunto de teste (`test.zip`)
2. Sortear o mesmo número de áudios por sotaque
3. Transcrever cada áudio com o Whisper
4. Normalizar os textos e calcular a WER
5. Comparar a WER entre os sotaques

**Antes de rodar:** crie um arquivo `.env` na raiz do projeto com `HF_TOKEN=seu_token` (veja `.env.example`).

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt
import pandas as pd

from src.download_coraa import download_audio
from src.sample import stratified_sample
from src.transcribe import transcribe_sample
from src.evaluate import evaluate_model

N_POR_SOTAQUE = 50       # piloto; para o trabalho final, aumentar para 300+
MODELO = "whisper-small"

## 1. Baixar os áudios do conjunto de teste
Só é preciso rodar uma vez. O arquivo é grande e pode demorar.

In [ ]:
zip_path = ROOT_DIR / "data" / "raw" / "coraa" / "test.zip"
if not zip_path.exists():
    download_audio("test")
print(zip_path, f"{zip_path.stat().st_size / 1e9:.1f} GB")

## 2. Amostra estratificada
Mesmo número de áudios por sotaque, com semente fixa (todos do grupo obtêm a mesma amostra).

In [ ]:
amostra = stratified_sample(split="test", n_per_accent=N_POR_SOTAQUE)
amostra["accent"].value_counts()

## 3. Transcrição
Na primeira execução o modelo é baixado (~1 GB). Em CPU leva alguns segundos por áudio. Se a execução parar, rode a célula de novo: ela continua de onde parou.

**Atenção:** se mudar os parâmetros de geração em `src/transcribe.py`, apague antes o arquivo `data/processed/transcricoes_whisper-small.parquet`, senão os áudios já transcritos são reaproveitados.

In [ ]:
transcricoes = transcribe_sample(amostra, model_name=MODELO, split="test")
transcricoes[["accent", "text", "hypothesis"]].head(10)

## 4. WER por sotaque
`wer` é a taxa de erro de palavra de cada sotaque; `ic95_inf` e `ic95_sup` formam o intervalo de confiança de 95%.

`n_alucinacoes` conta os áudios em que o modelo inventou mais palavras do que existiam na fala; `wer_sem_alucinacao` é a WER desconsiderando esses áudios.

In [ ]:
por_audio, por_sotaque = evaluate_model(ROOT_DIR / "data" / "processed" / f"transcricoes_{MODELO}.parquet")
por_sotaque.style.format({"wer": "{:.1%}", "ic95_inf": "{:.1%}", "ic95_sup": "{:.1%}", "wer_sem_alucinacao": "{:.1%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
dados = por_sotaque.sort_values("wer")
erro = [dados["wer"] - dados["ic95_inf"], dados["ic95_sup"] - dados["wer"]]
ax.barh(dados["accent"], dados["wer"] * 100, xerr=[e * 100 for e in erro], capsize=4, color="#4C72B0")
ax.set_xlabel("WER (%)")
ax.set_title(f"Taxa de erro de palavra por sotaque · {MODELO} · {N_POR_SOTAQUE} áudios por sotaque")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(ROOT_DIR / "results" / f"wer_por_sotaque_{MODELO}.png", dpi=150)
plt.show()

## 5. Exemplos de erros
Os piores casos de cada sotaque, para ilustrar na apresentação.

In [ ]:
pd.set_option("display.max_colwidth", None)
(por_audio.sort_values("wer", ascending=False)
    .groupby("accent").head(3)
    [["accent", "ref_norm", "hyp_norm", "wer"]]
    .sort_values("accent"))

## Cuidados ao interpretar

- Com 50 áudios por sotaque, intervalos de confiança que se sobrepõem indicam que a diferença **pode ser acaso**.
- Cada sotaque vem de um acervo diferente (NURC-Recife, SP-2010, ALIP, C-ORAL Brasil), com épocas, qualidade de gravação e tipos de entrevista diferentes. Parte da diferença de WER pode vir disso, e não do sotaque.